In [0]:
products = [
    (101, "Laptop", "Electronics", 60000),
    (102, "Mobile", "Electronics", 30000),
    (103, "Headphones", "Accessories", 3000),
    (104, "Keyboard", "Accessories", 2000),
    (105, "Monitor", "Electronics", 15000)
]

product_columns = [
    "ProductId",
    "ProductName",
    "Category",
    "Price"
]

products_df = spark.createDataFrame(
    products,
    product_columns
)

display(products_df)

ProductId,ProductName,Category,Price
101,Laptop,Electronics,60000
102,Mobile,Electronics,30000
103,Headphones,Accessories,3000
104,Keyboard,Accessories,2000
105,Monitor,Electronics,15000


In [0]:
products_df.write.mode("overwrite").saveAsTable("silver.products")

In [0]:
%sql
select * from silver.products;

ProductId,ProductName,Category,Price
101,Laptop,Electronics,60000
102,Mobile,Electronics,30000
103,Headphones,Accessories,3000
104,Keyboard,Accessories,2000
105,Monitor,Electronics,15000


In [0]:
orders = [
    (1001, 1, 101, 1, "2026-08-01"),
    (1002, 2, 102, 2, "2026-08-01"),
    (1003, 3, 103, 3, "2026-08-02"),
    (1004, 1, 104, 2, "2026-08-02"),
    (1005, 4, 105, 1, "2026-08-03"),
    (1006, 5, 101, 1, "2026-08-03"),
    (1007, 2, 103, 2, "2026-08-04")
]

order_columns = [
    "OrderId",
    "CustomerId",
    "ProductId",
    "Quantity",
    "OrderDate"
]

orders_df = spark.createDataFrame(
    orders,
    order_columns
)

display(orders_df)

OrderId,CustomerId,ProductId,Quantity,OrderDate
1001,1,101,1,2026-08-01
1002,2,102,2,2026-08-01
1003,3,103,3,2026-08-02
1004,1,104,2,2026-08-02
1005,4,105,1,2026-08-03
1006,5,101,1,2026-08-03
1007,2,103,2,2026-08-04


In [0]:
orders_df.write.mode("overwrite").saveAsTable("silver.orders")

In [0]:
%sql
select * from silver.orders;

OrderId,CustomerId,ProductId,Quantity,OrderDate
1001,1,101,1,2026-08-01
1002,2,102,2,2026-08-01
1003,3,103,3,2026-08-02
1004,1,104,2,2026-08-02
1005,4,105,1,2026-08-03
1006,5,101,1,2026-08-03
1007,2,103,2,2026-08-04


In [0]:
customers_df = spark.table("silver.customers")

products_df = spark.table("silver.products")

orders_df = spark.table("silver.orders")

In [0]:
order_customer_df=orders_df.join(customers_df,orders_df.CustomerId==customers_df.CustomerId,"inner")
display(order_customer_df)

OrderId,CustomerId,ProductId,Quantity,OrderDate,CustomerId,CustomerName,Email,City,Age
1006,5,101,1,2026-08-03,5,David,david@gmail.com,Coimbatore,40
1004,1,104,2,2026-08-02,1,Saravana,saravana@gmail.com,Chennai,30
1003,3,103,3,2026-08-02,3,Alice,alice@gmail.com,Hyderabad,25
1007,2,103,2,2026-08-04,2,John,john@gmail.com,Bangalore,28
1005,4,105,1,2026-08-03,4,Bob,bob@gmail.com,Chennai,35
1001,1,101,1,2026-08-01,1,Saravana,saravana@gmail.com,Chennai,30
1002,2,102,2,2026-08-01,2,John,john@gmail.com,Bangalore,28


In [0]:
orders_details_df= order_customer_df.join(products_df,order_customer_df.ProductId==products_df.ProductId,"inner")
display(orders_details_df)

OrderId,CustomerId,ProductId,Quantity,OrderDate,CustomerId,CustomerName,Email,City,Age,ProductId,ProductName,Category,Price
1001,1,101,1,2026-08-01,1,Saravana,saravana@gmail.com,Chennai,30,101,Laptop,Electronics,60000
1002,2,102,2,2026-08-01,2,John,john@gmail.com,Bangalore,28,102,Mobile,Electronics,30000
1003,3,103,3,2026-08-02,3,Alice,alice@gmail.com,Hyderabad,25,103,Headphones,Accessories,3000
1004,1,104,2,2026-08-02,1,Saravana,saravana@gmail.com,Chennai,30,104,Keyboard,Accessories,2000
1005,4,105,1,2026-08-03,4,Bob,bob@gmail.com,Chennai,35,105,Monitor,Electronics,15000
1006,5,101,1,2026-08-03,5,David,david@gmail.com,Coimbatore,40,101,Laptop,Electronics,60000
1007,2,103,2,2026-08-04,2,John,john@gmail.com,Bangalore,28,103,Headphones,Accessories,3000


In [0]:
order_details_df = orders_details_df.select(
    orders_df.OrderId,
    customers_df.CustomerId,
    customers_df.CustomerName,
    customers_df.City,
    products_df.ProductId,
    products_df.ProductName,
    products_df.Category,
    products_df.Price,
    orders_df.Quantity,
    orders_df.OrderDate
)

In [0]:

from pyspark.sql.functions import col

order_details_df=order_details_df.withColumn("OrderAmount",col("Quantity")*col("Price"))
display(order_details_df)

OrderId,CustomerId,CustomerName,City,ProductId,ProductName,Category,Price,Quantity,OrderDate,OrderAmount
1001,1,Saravana,Chennai,101,Laptop,Electronics,60000,1,2026-08-01,60000
1002,2,John,Bangalore,102,Mobile,Electronics,30000,2,2026-08-01,60000
1003,3,Alice,Hyderabad,103,Headphones,Accessories,3000,3,2026-08-02,9000
1004,1,Saravana,Chennai,104,Keyboard,Accessories,2000,2,2026-08-02,4000
1005,4,Bob,Chennai,105,Monitor,Electronics,15000,1,2026-08-03,15000
1006,5,David,Coimbatore,101,Laptop,Electronics,60000,1,2026-08-03,60000
1007,2,John,Bangalore,103,Headphones,Accessories,3000,2,2026-08-04,6000


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
order_details_df.write.mode("overwrite").saveAsTable("gold.order_details")

In [0]:
%sql
SELECT *
FROM gold.order_details;

OrderId,CustomerId,CustomerName,City,ProductId,ProductName,Category,Price,Quantity,OrderDate,OrderAmount
1001,1,Saravana,Chennai,101,Laptop,Electronics,60000,1,2026-08-01,60000
1002,2,John,Bangalore,102,Mobile,Electronics,30000,2,2026-08-01,60000
1003,3,Alice,Hyderabad,103,Headphones,Accessories,3000,3,2026-08-02,9000
1004,1,Saravana,Chennai,104,Keyboard,Accessories,2000,2,2026-08-02,4000
1005,4,Bob,Chennai,105,Monitor,Electronics,15000,1,2026-08-03,15000
1006,5,David,Coimbatore,101,Laptop,Electronics,60000,1,2026-08-03,60000
1007,2,John,Bangalore,103,Headphones,Accessories,3000,2,2026-08-04,6000


In [0]:
from pyspark.sql.functions import sum
customer_revenue_df=order_details_df.groupBy("CustomerName").agg(sum("OrderAmount").alias("TotalRevenue"))
display(customer_revenue_df)

CustomerName,TotalRevenue
Saravana,64000
Bob,15000
David,60000
Alice,9000
John,66000


In [0]:
customer_revenue_df.write \
    .mode("overwrite") \
    .saveAsTable("gold.customer_revenue")